Summarization creates a shorter version of a document or an article that captures all the important information. Along with translation, it is another example of a task that can be formulated as a sequence-to-sequence task. Summarization can be:

- Extractive: extract the most relevant information from a document.
- Abstractive: generate new text that captures the most relevant information.

This guide will show you how to:

1. Finetune [T5](https://huggingface.co/google-t5/t5-small) on the California state bill subset of the [BillSum](https://huggingface.co/datasets/billsum) dataset for abstractive summarization.
2. Use your finetuned model for inference.

In [ ]:
! pip install transformers datasets evaluate accelerate huggingface_hub

In [ ]:
user_name = "amin-oj"

from huggingface_hub import notebook_login
notebook_login()

## Load BillSum dataset

In [ ]:
from datasets import load_dataset
billsum = load_dataset("billsum", split="ca_test")
# loading the smaller California state bill subset
billsum = billsum.train_test_split(test_size=0.2)
exm = billsum["train"][0]
print(type(exm))
print(exm.keys())

There are two fields that you'll want to use:

- `text`: the text of the bill which'll be the input to the model.
- `summary`: a condensed version of `text` which'll be the model target.

## Preprocess

In [ ]:
from transformers import AutoTokenizer
checkpoint = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def preprocess_function(examples, prefix):
    # Prefix the input with a prompt so T5 knows
    # this is a summarization task.
    # Some models capable of multiple NLP tasks
    # require prompting for specific tasks.
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = tokenizer(
        inputs,
        max_length=1024,
        truncation=True
    )

    # Use the keyword `text_target` argument when tokenizing labels.
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_billsum = billsum.map(
    preprocess_function,
    batched=True,
    fn_kwargs={"prefix": "summarize: "}
)

In [ ]:
from transformers import DataCollatorForSeq2Seq
# It's more efficient to *dynamically pad* the sentences to the longest length
# in a batch during collation, instead of padding the whole dataset to
# the maximum length.
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

## Evaluate

In [ ]:
!pip install rouge_score

In [ ]:
import numpy as np
import evaluate
rouge = evaluate.load("rouge")

Then create a function that passes your predictions and labels to [compute](https://huggingface.co/docs/evaluate/main/en/package_reference/main_classes#evaluate.EvaluationModule.compute) to calculate the ROUGE metric:

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

## Train

In [ ]:
from transformers import (AutoModelForSeq2SeqLM,
                          Seq2SeqTrainingArguments,
                          Seq2SeqTrainer)

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

model_name = checkpoint.split("/")[-1]
task = "summarization"
data_id = "billsum"
ckpt_name = f"{model_name}-finetuned-{task}-{data_id}"

training_args = Seq2SeqTrainingArguments(
    output_dir=ckpt_name,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True, #change to bf16=True for XPU
    push_to_hub=True,
    report_to = 'none'
    # to disable w&b
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_billsum["train"],
    eval_dataset=tokenized_billsum["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.push_to_hub()

## Inference

For T5, you need to prefix your input depending on the task you're working on. For summarization you should prefix your input as shown below:

In [ ]:
text = """
summarize: The Inflation Reduction Act lowers prescription drug costs,
health care costs, and energy costs. It's the most aggressive action
on tackling the climate crisis in American history, which will lift up
American workers and create good-paying, union jobs across the country.
It'll lower the deficit and ask the ultra-wealthy and corporations
to pay their fair share. And no one making under $400,000 per year
will pay a penny more in taxes."""

The simplest way to try out your finetuned model for inference is to use it in a [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline). Instantiate a `pipeline` for summarization with your model, and pass your text to it:

In [ ]:
from transformers import pipeline
summarizer = pipeline("summarization", model=f"{user_name}/{ckpt_name}")
summarizer(text)

You can also manually replicate the results of the `pipeline` if you'd like:

Tokenize the text and return the `input_ids` as PyTorch tensors:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(f"{user_name}/{ckpt_name}")
inputs = tokenizer(text, return_tensors="pt").input_ids

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(f"{user_name}/{ckpt_name}")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=False)

tokenizer.decode(outputs[0], skip_special_tokens=True)

For more details about the different text generation strategies and parameters for controlling generation, check out the [Text Generation](https://huggingface.co/docs/transformers/main/en/tasks/../main_classes/text_generation) API.